# Batch Document-Aware Extraction with Llama Vision

Production-ready batch processing notebook for document extraction using Llama-3.2-Vision-Instruct.

**Key Features:**
- Early model loading for efficient batch processing
- Configurable output directory (supports external paths)
- Automatic document type detection for each image
- Comprehensive accuracy tracking and runtime metrics
- Professional visualizations with seaborn and matplotlib
- Pandas DataFrames for detailed analysis
- V100 GPU optimization with ResilientGenerator
- Automatic export of results to CSV, JSON, and PNG formats

# Core Library Imports

In [1]:
# Standard library imports
import os
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Data processing and visualization
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

# Rich console for progress tracking
from rich import print as rprint
from rich.console import Console
from rich.table import Table
from rich.progress import track

warnings.filterwarnings('ignore')

console = Console()
rprint("[bold green]✅ Core libraries imported successfully[/bold green]")

✅ Core libraries imported successfully

# Configuration & Output Directory Setup

In [2]:
# =============================================================================
# OUTPUT DIRECTORY CONFIGURATION
# =============================================================================

# Check for environment variable first, then use default
OUTPUT_BASE = os.getenv('OUTPUT_DIR', 'output')

# Convert to Path and make absolute if relative
OUTPUT_BASE = Path(OUTPUT_BASE)
if not OUTPUT_BASE.is_absolute():
    OUTPUT_BASE = Path.cwd() / OUTPUT_BASE

# Create timestamp for this batch run
BATCH_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create subdirectories
OUTPUT_DIRS = {
    'base': OUTPUT_BASE,
    'batch': OUTPUT_BASE / 'batch_results',
    'csv': OUTPUT_BASE / 'csv',
    'visualizations': OUTPUT_BASE / 'visualizations',
    'reports': OUTPUT_BASE / 'reports'
}

# Create all directories
for dir_name, dir_path in OUTPUT_DIRS.items():
    dir_path.mkdir(parents=True, exist_ok=True)
    
rprint("[bold blue]📁 Output Directory Configuration[/bold blue]")
rprint(f"[cyan]Base output directory: {OUTPUT_BASE}[/cyan]")
rprint(f"[cyan]Batch timestamp: {BATCH_TIMESTAMP}[/cyan]")
rprint("[green]✅ Output directories created[/green]")

# Display directory structure
table = Table(title="Output Directory Structure")
table.add_column("Directory", style="cyan")
table.add_column("Path", style="yellow")
for name, path in OUTPUT_DIRS.items():
    if name != 'base':
        table.add_row(name, str(path.relative_to(OUTPUT_BASE)))
console.print(table)

📁 Output Directory Configuration

Base output directory: /home/jovyan/nfs_share/tod/LMM_POC/output

Batch timestamp: 20250910_044415

✅ Output directories created

    Output Directory Structure     
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Directory      ┃ Path           ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ batch          │ batch_results  │
│ csv            │ csv            │
│ visualizations │ visualizations │
│ reports        │ reports        │
└────────────────┴────────────────┘

# Model Configuration & Early Loading

In [3]:
# =============================================================================
# MODEL CONFIGURATION
# =============================================================================

# Model path
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Batch processing configuration
BATCH_CONFIG = {
    'data_dir': 'evaluation_data',
    'ground_truth': 'evaluation_data/ground_truth.csv',
    'max_images': None,  # None for all images, or set a number to limit
    'document_types': None,  # None for all types, or list like ['invoice', 'receipt']
}

# Prompt configuration files
PROMPT_CONFIG = {
    'detection_file': 'prompts/document_type_detection.yaml',
    'detection_key': 'detection',
    'extraction_files': {
        'INVOICE': 'prompts/invoice_extraction.yaml',
        'RECEIPT': 'prompts/receipt_extraction.yaml',
        'BANK_STATEMENT': 'prompts/bank_statement_extraction.yaml'
    },
    'extraction_keys': {
        'INVOICE': 'standard',
        'RECEIPT': 'standard',
        'BANK_STATEMENT': 'flat'
    }
}

# V100 optimization settings
V100_CONFIG = {
    'use_quantization': True,
    'device_map': 'auto',
    'max_new_tokens': 4000,
    'torch_dtype': 'bfloat16',
    'low_cpu_mem_usage': True
}

rprint("[bold blue]🚀 Model Configuration[/bold blue]")
rprint(f"[cyan]Model: {Path(MODEL_PATH).name}[/cyan]")
rprint(f"[cyan]Data directory: {BATCH_CONFIG['data_dir']}[/cyan]")
rprint("[yellow]V100 optimizations: Enabled[/yellow]")

🚀 Model Configuration

Model: Llama-3.2-11B-Vision-Instruct

Data directory: evaluation_data

V100 optimizations: Enabled

In [4]:
# =============================================================================
# EARLY MODEL LOADING - Load once for entire batch
# =============================================================================

from common.model_loader import load_v100_model

rprint("[bold green]🔧 Loading Llama Vision model with V100 optimizations...[/bold green]")

# Load model with V100 optimizations
model, processor = load_v100_model(
    model_path=MODEL_PATH,
    use_quantization=V100_CONFIG['use_quantization'],
    device_map=V100_CONFIG['device_map'],
    max_new_tokens=V100_CONFIG['max_new_tokens'],
    torch_dtype=V100_CONFIG['torch_dtype'],
    low_cpu_mem_usage=V100_CONFIG['low_cpu_mem_usage']
)

rprint("[bold green]✅ Model loaded and ready for batch processing![/bold green]")

🔧 Loading Llama Vision model with V100 optimizations...

🚀 Loading Llama Vision model with V100 production optimizations...

🔧 Configuring V100-optimized CUDA memory allocation...

🔧 CUDA memory allocation configured: max_split_size_mb:64
💡 Using 64MB memory blocks to reduce fragmentation
📊 Initial CUDA state: Allocated=0.00GB, Reserved=0.00GB


🔧 Configuring V100-optimized 8-bit quantization with BitsAndBytesConfig

...

Performing initial V100 memory cleanup...

🧹 Clearing model caches...
✅ Model caches cleared
🧹 Memory state: Allocated=4.70GB, Reserved=4.75GB, Fragmentation=0.04GB


🎉 V100-optimized model loading and validation complete!

🔧 V100 optimizations active: CPU offload, vision skip, 32MB blocks

✅ Model loaded and ready for batch processing!

In [5]:
# =============================================================================
# INITIALIZE EXTRACTOR WITH LOADED MODEL
# =============================================================================

from common.llama_vision_table_extractor import LlamaVisionTableExtractor

# Initialize the V100-optimized extractor with existing model
extractor = LlamaVisionTableExtractor(processor=processor, model=model)

rprint("[green]✅ V100-optimized extractor initialized[/green]")
rprint("[cyan]Features: ResilientGenerator, memory cleanup, fragmentation detection[/cyan]")

✅ Using existing model and processor with V100 ResilientGenerator

✅ V100-optimized extractor initialized

Features: ResilientGenerator, memory cleanup, fragmentation detection

# Batch Image Discovery & Setup

In [ ]:
# =============================================================================
# DISCOVER IMAGES FOR BATCH PROCESSING
# =============================================================================

from common.extraction_parser import discover_images
from common.evaluation_metrics import load_ground_truth

# Discover all images
all_images = discover_images(BATCH_CONFIG['data_dir'])
rprint(f"[cyan]📷 Found {len(all_images)} images in {BATCH_CONFIG['data_dir']}[/cyan]")

# Load ground truth
ground_truth = load_ground_truth(BATCH_CONFIG['ground_truth'])
rprint(f"[cyan]📊 Loaded ground truth with {len(ground_truth)} entries[/cyan]")

# Filter by document type if specified
if BATCH_CONFIG['document_types']:
    filtered_images = []
    for img in all_images:
        img_name = Path(img).name
        if img_name in ground_truth:
            doc_type = ground_truth[img_name].get('DOCUMENT_TYPE', '').lower()
            if any(dt.lower() in doc_type for dt in BATCH_CONFIG['document_types']):
                filtered_images.append(img)
    all_images = filtered_images
    rprint(f"[yellow]🔍 Filtered to {len(all_images)} images of types: {BATCH_CONFIG['document_types']}[/yellow]")

# Limit images if specified
if BATCH_CONFIG['max_images']:
    all_images = all_images[:BATCH_CONFIG['max_images']]
    rprint(f"[yellow]📸 Limited to {len(all_images)} images[/yellow]")

# Display image list
rprint(f"\n[bold green]Ready to process {len(all_images)} images[/bold green]")
if len(all_images) <= 10:
    for img in all_images:
        rprint(f"  • {Path(img).name}")
else:
    rprint(f"  • {Path(all_images[0]).name}")
    rprint(f"  • ...")
    rprint(f"  • {Path(all_images[-1]).name}")

# Batch Processing with Progress Tracking

In [ ]:
# =============================================================================
# BATCH PROCESSING WITH DOCUMENT-AWARE EXTRACTION
# =============================================================================

from common.document_detector import detect_document_type
from common.prompt_loader import load_document_prompt
from common.ground_truth_evaluator import GroundTruthEvaluator

# Initialize results storage
batch_results = []
processing_times = []
document_types_found = {}

# Initialize ground truth evaluator
evaluator = GroundTruthEvaluator(BATCH_CONFIG['ground_truth'])

rprint("\n[bold blue]🚀 Starting Batch Processing[/bold blue]")
console.rule("[bold green]Batch Extraction[/bold green]")

# Process each image
for idx, image_path in enumerate(track(all_images, description="Processing images..."), 1):
    image_name = Path(image_path).name
    
    try:
        # Record start time
        start_time = time.time()
        
        # Step 1: Detect document type
        document_type = detect_document_type(
            image_path=image_path,
            extractor=extractor,
            detection_prompt_file=PROMPT_CONFIG['detection_file'],
            prompt_files=PROMPT_CONFIG['extraction_files'],
            detection_prompt_key=PROMPT_CONFIG['detection_key'],
            verbose=False  # Suppress individual outputs for batch
        )
        
        # Track document types
        document_types_found[document_type] = document_types_found.get(document_type, 0) + 1
        
        # Step 2: Load document-specific prompt
        extraction_prompt, prompt_name, _ = load_document_prompt(
            prompt_files=PROMPT_CONFIG['extraction_files'],
            prompt_keys=PROMPT_CONFIG['extraction_keys'],
            document_type=document_type,
            verbose=False
        )
        
        # Step 3: Extract fields
        extraction_result = extractor.test_extraction(
            image_path, 
            extraction_prompt,
            verbose=False
        )
        
        # Step 4: Evaluate against ground truth
        evaluation = evaluator.evaluate_extraction(
            test_result=extraction_result,
            document_type=document_type,
            image_path=image_path,
            verbose=False
        )
        
        # Record processing time
        processing_time = time.time() - start_time
        processing_times.append(processing_time)
        
        # Store results
        result = {
            'image_name': image_name,
            'image_path': image_path,
            'document_type': document_type,
            'extraction_result': extraction_result,
            'evaluation': evaluation,
            'processing_time': processing_time,
            'prompt_used': prompt_name,
            'timestamp': datetime.now().isoformat()
        }
        batch_results.append(result)
        
        # Quick status update
        if idx % 5 == 0 or idx == len(all_images):
            accuracy = evaluation.get('overall_accuracy', 0) * 100 if evaluation else 0
            rprint(f"  [{idx}/{len(all_images)}] {image_name}: {document_type} - Accuracy: {accuracy:.1f}% - Time: {processing_time:.2f}s")
            
    except Exception as e:
        rprint(f"[red]❌ Error processing {image_name}: {e}[/red]")
        # Store error result
        batch_results.append({
            'image_name': image_name,
            'image_path': image_path,
            'error': str(e),
            'processing_time': time.time() - start_time
        })

# Summary
console.rule("[bold green]Batch Processing Complete[/bold green]")
rprint(f"\n[bold green]✅ Processed {len(batch_results)} images[/bold green]")
rprint(f"[cyan]Average processing time: {np.mean(processing_times):.2f}s[/cyan]")
rprint(f"[cyan]Document types found: {document_types_found}[/cyan]")

# Create Comprehensive DataFrames

In [ ]:
# =============================================================================
# CREATE PANDAS DATAFRAMES FOR ANALYSIS
# =============================================================================

# Create main results DataFrame
results_data = []
for result in batch_results:
    if 'error' not in result:
        row = {
            'image_name': result['image_name'],
            'document_type': result['document_type'],
            'overall_accuracy': result['evaluation'].get('overall_accuracy', 0) * 100,
            'fields_extracted': result['evaluation'].get('fields_extracted', 0),
            'fields_matched': result['evaluation'].get('fields_matched', 0),
            'total_fields': result['evaluation'].get('total_fields', 0),
            'processing_time': result['processing_time'],
            'prompt_used': result['prompt_used']
        }
        
        # Add individual field accuracies
        if 'field_accuracies' in result['evaluation']:
            for field, data in result['evaluation']['field_accuracies'].items():
                row[f'accuracy_{field}'] = data.get('accuracy', 0) * 100
                
        results_data.append(row)

df_results = pd.DataFrame(results_data)

# Create summary statistics DataFrame
summary_stats = {
    'Total Images': len(batch_results),
    'Successful Extractions': len(results_data),
    'Failed Extractions': len(batch_results) - len(results_data),
    'Average Accuracy (%)': df_results['overall_accuracy'].mean() if len(df_results) > 0 else 0,
    'Median Accuracy (%)': df_results['overall_accuracy'].median() if len(df_results) > 0 else 0,
    'Min Accuracy (%)': df_results['overall_accuracy'].min() if len(df_results) > 0 else 0,
    'Max Accuracy (%)': df_results['overall_accuracy'].max() if len(df_results) > 0 else 0,
    'Average Processing Time (s)': np.mean(processing_times),
    'Total Processing Time (s)': sum(processing_times),
    'Throughput (images/min)': 60 / np.mean(processing_times) if processing_times else 0
}

df_summary = pd.DataFrame([summary_stats]).T
df_summary.columns = ['Value']

# Create document type statistics DataFrame
if len(df_results) > 0:
    df_doctype_stats = df_results.groupby('document_type').agg({
        'overall_accuracy': ['mean', 'median', 'std', 'min', 'max'],
        'processing_time': ['mean', 'median'],
        'image_name': 'count'
    }).round(2)
    df_doctype_stats.columns = ['_'.join(col).strip() for col in df_doctype_stats.columns.values]
    df_doctype_stats.rename(columns={'image_name_count': 'count'}, inplace=True)
else:
    df_doctype_stats = pd.DataFrame()

# Display DataFrames
rprint("\n[bold blue]📊 Results Summary[/bold blue]")
display(df_summary)

if not df_doctype_stats.empty:
    rprint("\n[bold blue]📈 Document Type Statistics[/bold blue]")
    display(df_doctype_stats)

# Save DataFrames to CSV
csv_prefix = OUTPUT_DIRS['csv'] / f"batch_{BATCH_TIMESTAMP}"
df_results.to_csv(f"{csv_prefix}_results.csv", index=False)
df_summary.to_csv(f"{csv_prefix}_summary.csv")
if not df_doctype_stats.empty:
    df_doctype_stats.to_csv(f"{csv_prefix}_doctype_stats.csv")

rprint(f"[green]✅ DataFrames saved to {OUTPUT_DIRS['csv']}[/green]")

# Generate Seaborn Visualizations

In [ ]:
# =============================================================================
# CREATE COMPREHENSIVE VISUALIZATIONS
# =============================================================================

if len(df_results) > 0:
    # Set style
    plt.style.use('default')
    sns.set_palette("husl")
    
    # Create 2x2 dashboard
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Llama Vision Batch Processing Dashboard - {BATCH_TIMESTAMP}', fontsize=16, fontweight='bold')
    
    # 1. Accuracy Distribution Histogram
    sns.histplot(data=df_results, x='overall_accuracy', bins=20, kde=True, ax=ax1, color='skyblue')
    ax1.axvline(x=80, color='orange', linestyle='--', label='Pilot Ready (80%)')
    ax1.axvline(x=95, color='green', linestyle='--', label='Production Ready (95%)')
    ax1.set_title('Accuracy Distribution', fontweight='bold')
    ax1.set_xlabel('Overall Accuracy (%)')
    ax1.set_ylabel('Count')
    ax1.legend()
    
    # 2. Processing Time by Document Type
    if 'document_type' in df_results.columns and df_results['document_type'].nunique() > 1:
        sns.boxplot(data=df_results, x='document_type', y='processing_time', ax=ax2)
        ax2.set_title('Processing Time by Document Type', fontweight='bold')
        ax2.set_xlabel('Document Type')
        ax2.set_ylabel('Processing Time (seconds)')
        ax2.tick_params(axis='x', rotation=45)
    else:
        sns.histplot(data=df_results, x='processing_time', bins=15, kde=True, ax=ax2, color='coral')
        ax2.set_title('Processing Time Distribution', fontweight='bold')
        ax2.set_xlabel('Processing Time (seconds)')
        ax2.set_ylabel('Count')
    
    # 3. Accuracy vs Processing Time Scatter
    scatter = ax3.scatter(df_results['processing_time'], 
                         df_results['overall_accuracy'],
                         c=df_results['overall_accuracy'],
                         cmap='RdYlGn', 
                         alpha=0.6,
                         s=100)
    ax3.set_title('Accuracy vs Processing Time', fontweight='bold')
    ax3.set_xlabel('Processing Time (seconds)')
    ax3.set_ylabel('Overall Accuracy (%)')
    plt.colorbar(scatter, ax=ax3, label='Accuracy (%)')
    
    # 4. Document Type Performance Summary
    if not df_doctype_stats.empty:
        doc_perf = df_results.groupby('document_type')['overall_accuracy'].mean().sort_values(ascending=True)
        colors = ['red' if acc < 60 else 'orange' if acc < 80 else 'green' for acc in doc_perf.values]
        doc_perf.plot(kind='barh', ax=ax4, color=colors)
        ax4.set_title('Average Accuracy by Document Type', fontweight='bold')
        ax4.set_xlabel('Average Accuracy (%)')
        ax4.set_ylabel('Document Type')
        ax4.axvline(x=80, color='orange', linestyle='--', alpha=0.5)
        ax4.axvline(x=95, color='green', linestyle='--', alpha=0.5)
        
        # Add value labels
        for i, (idx, val) in enumerate(doc_perf.items()):
            ax4.text(val + 1, i, f'{val:.1f}%', va='center')
    else:
        # Field extraction success rate
        if 'fields_matched' in df_results.columns:
            field_success = (df_results['fields_matched'] / df_results['total_fields'] * 100).mean()
            ax4.bar(['Field Extraction\nSuccess Rate'], [field_success], color='steelblue')
            ax4.set_ylabel('Success Rate (%)')
            ax4.set_title('Overall Field Extraction Performance', fontweight='bold')
            ax4.set_ylim(0, 100)
            ax4.text(0, field_success + 2, f'{field_success:.1f}%', ha='center')
    
    plt.tight_layout()
    
    # Save dashboard
    dashboard_path = OUTPUT_DIRS['visualizations'] / f"dashboard_{BATCH_TIMESTAMP}.png"
    plt.savefig(dashboard_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    rprint(f"[green]✅ Dashboard saved to {dashboard_path}[/green]")
else:
    rprint("[yellow]⚠️ No successful extractions to visualize[/yellow]")

# Field-Level Accuracy Heatmap

In [ ]:
# =============================================================================
# FIELD-LEVEL ACCURACY HEATMAP
# =============================================================================

if len(df_results) > 0:
    # Extract field accuracy columns
    field_cols = [col for col in df_results.columns if col.startswith('accuracy_')]
    
    if field_cols:
        # Create field accuracy matrix
        field_accuracy_data = df_results[['image_name'] + field_cols].set_index('image_name')
        field_accuracy_data.columns = [col.replace('accuracy_', '') for col in field_accuracy_data.columns]
        
        # Calculate average accuracy per field
        field_avg = field_accuracy_data.mean().sort_values(ascending=False)
        
        # Create heatmap
        plt.figure(figsize=(14, max(8, len(df_results) * 0.3)))
        
        # Reorder columns by average accuracy
        field_accuracy_ordered = field_accuracy_data[field_avg.index]
        
        # Create heatmap
        sns.heatmap(field_accuracy_ordered.T, 
                   cmap='RdYlGn',
                   vmin=0, vmax=100,
                   annot=False,
                   fmt='.0f',
                   cbar_kws={'label': 'Accuracy (%)'},
                   linewidths=0.5)
        
        plt.title(f'Field-Level Accuracy Heatmap - {BATCH_TIMESTAMP}', fontsize=14, fontweight='bold')
        plt.xlabel('Document')
        plt.ylabel('Field')
        plt.xticks(rotation=90)
        plt.tight_layout()
        
        # Save heatmap
        heatmap_path = OUTPUT_DIRS['visualizations'] / f"field_accuracy_heatmap_{BATCH_TIMESTAMP}.png"
        plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        rprint(f"[green]✅ Heatmap saved to {heatmap_path}[/green]")
        
        # Display field statistics
        rprint("\n[bold blue]📊 Field Accuracy Statistics[/bold blue]")
        field_stats = pd.DataFrame({
            'Average Accuracy (%)': field_avg,
            'Min Accuracy (%)': field_accuracy_data.min(),
            'Max Accuracy (%)': field_accuracy_data.max(),
            'Std Dev (%)': field_accuracy_data.std()
        }).round(2)
        display(field_stats.head(10))  # Show top 10 fields
        
        # Save field statistics
        field_stats.to_csv(OUTPUT_DIRS['csv'] / f"batch_{BATCH_TIMESTAMP}_field_stats.csv")
    else:
        rprint("[yellow]⚠️ No field-level accuracy data available[/yellow]")

# Generate Executive Summary Report

In [ ]:
# =============================================================================
# GENERATE EXECUTIVE SUMMARY REPORT
# =============================================================================

# Calculate key metrics
total_images = len(batch_results)
successful_extractions = len([r for r in batch_results if 'error' not in r])
avg_accuracy = df_results['overall_accuracy'].mean() if len(df_results) > 0 else 0
total_time = sum(processing_times)
throughput = 60 / np.mean(processing_times) if processing_times else 0

# Determine deployment readiness
if avg_accuracy >= 95:
    readiness = "✅ **Production Ready**"
    readiness_color = "green"
elif avg_accuracy >= 80:
    readiness = "🟡 **Pilot Ready**"
    readiness_color = "yellow"
else:
    readiness = "🔴 **Needs Improvement**"
    readiness_color = "red"

# Create markdown report
report = f"""# Llama Vision Batch Processing Report

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Batch ID:** {BATCH_TIMESTAMP}  
**Model:** Llama-3.2-11B-Vision-Instruct  

## Executive Summary

### Overall Performance
- **Total Images Processed:** {total_images}
- **Successful Extractions:** {successful_extractions} ({successful_extractions/total_images*100:.1f}%)
- **Average Accuracy:** {avg_accuracy:.2f}%
- **Deployment Status:** {readiness}

### Processing Efficiency
- **Total Processing Time:** {total_time:.2f} seconds ({total_time/60:.1f} minutes)
- **Average Time per Image:** {np.mean(processing_times):.2f} seconds
- **Throughput:** {throughput:.1f} images/minute

### Document Type Distribution
"""

for doc_type, count in document_types_found.items():
    percentage = (count / total_images) * 100
    report += f"- **{doc_type}:** {count} ({percentage:.1f}%)\n"

# Add accuracy breakdown if available
if not df_doctype_stats.empty:
    report += "\n### Accuracy by Document Type\n"
    for doc_type in df_doctype_stats.index:
        mean_acc = df_doctype_stats.loc[doc_type, 'overall_accuracy_mean']
        report += f"- **{doc_type}:** {mean_acc:.2f}%\n"

# Add top performing images
if len(df_results) > 0:
    top_performers = df_results.nlargest(5, 'overall_accuracy')[['image_name', 'overall_accuracy', 'document_type']]
    report += "\n### Top Performing Images\n"
    for _, row in top_performers.iterrows():
        report += f"- {row['image_name']}: {row['overall_accuracy']:.1f}% ({row['document_type']})\n"

# Add areas for improvement
if len(df_results) > 0:
    poor_performers = df_results.nsmallest(5, 'overall_accuracy')[['image_name', 'overall_accuracy', 'document_type']]
    if poor_performers['overall_accuracy'].min() < 80:
        report += "\n### Areas for Improvement\n"
        for _, row in poor_performers.iterrows():
            report += f"- {row['image_name']}: {row['overall_accuracy']:.1f}% ({row['document_type']})\n"

report += f"""\n## Output Files Generated

All results have been saved to: `{OUTPUT_BASE}`

- **CSV Files:** `csv/batch_{BATCH_TIMESTAMP}_*.csv`
- **Visualizations:** `visualizations/*_{BATCH_TIMESTAMP}.png`
- **Full Report:** `reports/batch_report_{BATCH_TIMESTAMP}.md`

## Technical Details

- **V100 Optimizations:** Enabled (ResilientGenerator, Memory Cleanup)
- **Quantization:** 8-bit with BitsAndBytesConfig
- **Max Tokens:** 4000
- **Device:** CUDA (auto-mapped)
"""

# Display report
from IPython.display import Markdown
display(Markdown(report))

# Save report
report_path = OUTPUT_DIRS['reports'] / f"batch_report_{BATCH_TIMESTAMP}.md"
with open(report_path, 'w') as f:
    f.write(report)

rprint(f"[green]✅ Executive summary saved to {report_path}[/green]")

# Export Complete Results to JSON

In [ ]:
# =============================================================================
# EXPORT COMPLETE RESULTS TO JSON
# =============================================================================

import json

# Prepare complete results for JSON export
export_data = {
    'metadata': {
        'batch_id': BATCH_TIMESTAMP,
        'timestamp': datetime.now().isoformat(),
        'model': MODEL_PATH,
        'total_images': total_images,
        'successful_extractions': successful_extractions,
        'average_accuracy': avg_accuracy,
        'total_processing_time': total_time,
        'deployment_status': readiness.replace('*', '').strip()
    },
    'configuration': {
        'data_directory': BATCH_CONFIG['data_dir'],
        'ground_truth': BATCH_CONFIG['ground_truth'],
        'max_images': BATCH_CONFIG['max_images'],
        'document_types': BATCH_CONFIG['document_types'],
        'v100_optimizations': V100_CONFIG
    },
    'summary_statistics': summary_stats,
    'document_type_distribution': document_types_found,
    'results': []
}

# Add individual results (excluding large extraction data)
for result in batch_results:
    if 'error' not in result:
        export_result = {
            'image_name': result['image_name'],
            'document_type': result['document_type'],
            'prompt_used': result['prompt_used'],
            'processing_time': result['processing_time'],
            'evaluation_summary': {
                'overall_accuracy': result['evaluation'].get('overall_accuracy', 0),
                'fields_extracted': result['evaluation'].get('fields_extracted', 0),
                'fields_matched': result['evaluation'].get('fields_matched', 0),
                'total_fields': result['evaluation'].get('total_fields', 0)
            }
        }
        export_data['results'].append(export_result)

# Save JSON report
json_path = OUTPUT_DIRS['batch'] / f"batch_results_{BATCH_TIMESTAMP}.json"
with open(json_path, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

rprint(f"[green]✅ Complete results exported to {json_path}[/green]")

# Display final summary
console.rule("[bold green]Batch Processing Complete[/bold green]")

summary_table = Table(title="Final Summary")
summary_table.add_column("Metric", style="cyan")
summary_table.add_column("Value", style="green")

summary_table.add_row("Total Images", str(total_images))
summary_table.add_row("Success Rate", f"{(successful_extractions/total_images*100):.1f}%")
summary_table.add_row("Average Accuracy", f"{avg_accuracy:.2f}%")
summary_table.add_row("Throughput", f"{throughput:.1f} img/min")
summary_table.add_row("Output Directory", str(OUTPUT_BASE))

console.print(summary_table)

rprint(f"\n[bold green]🎉 All results saved to: {OUTPUT_BASE}[/bold green]")